# F1 Visor · Training workflow

This notebook runs the same scripts used by the GitHub project: dataset validation → training → evaluation → export. It does not define another model or trigger hardware. Execute each stage deliberately after inspecting its inputs. See `docs/PROJECT_OUTLINE.md` and `docs/TRAINING.md`.

Install `requirements-notebook.txt` in the active kernel environment before starting. Keep the historical V1/V2/V3 metrics in `docs/HISTORY.md` separate from this run.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT = next((p.resolve() for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "f1-visor")
                if (p / "f1visor").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("Open this notebook from the f1-visor project")
print("Project:", PROJECT)


## 1. Configure an experiment
Use manually labeled, permitted images. Choose a new RUN and MODEL path for each experiment. 224 pixels matches the intended Pi deployment resolution; 384 remains available for a separate comparison.

In [ ]:
RAW = PROJECT / "data" / "raw"
SPLIT = PROJECT / "data" / "split"
RUN = PROJECT / "runs" / "pi-224-01"
MODEL = PROJECT / "models" / "pi-224-01.pt"
IMAGE_SIZE = 224


## 2. Prepare data
Run only for a new split. If a reviewed split already exists, skip this cell. For related video frames, assign whole capture sessions to splits first; do not randomly mix neighboring frames.

In [ ]:
subprocess.run([sys.executable, str(PROJECT / "prepare_data.py"), "--source", str(RAW), "--output", str(SPLIT)], cwd=PROJECT, check=True)

## 3. Train
The first standard run downloads pretrained ImageNet weights. Checkpoint selection and threshold selection use validation data; test data is held out.

In [ ]:
subprocess.run([sys.executable, str(PROJECT / "train.py"), "--data", str(SPLIT), "--output", str(RUN), "--image-size", str(IMAGE_SIZE)], cwd=PROJECT, check=True)

## 4. Review test evidence
This generates curves, confusion matrices and hard examples using the saved validation threshold.

In [ ]:
subprocess.run([sys.executable, str(PROJECT / "evaluation_plots.py"), "--run", str(RUN), "--data", str(SPLIT)], cwd=PROJECT, check=True)

## 5. Export a matched model pair

In [ ]:
subprocess.run([sys.executable, str(PROJECT / "export_model.py"), "--checkpoint", str(RUN / "best.pt"), "--output", str(MODEL)], cwd=PROJECT, check=True)

## 6. Test and deploy
Test the exported `.pt` and `.json` pair on held-out clean/mud/water images using `detect_video.py`, then follow `docs/HARDWARE.md`. Set `config.local.json` → `model` to this exported model. Do not retain the old model path or borrow a threshold from another run. No cell in this notebook moves the servo.